# PCA vs. Factor Analysis: Complete Guide

## 1. The Core Philosophical Difference
- **Principal Component Analysis (PCA)** and **Factor Analysis (FA)** are often confused, but they serve fundamentally different purposes.
- **PCA is a Mathematical Compressor.** It seeks to maximize the **Total Variance** captured. It treats all variance (both signal and noise) as information to be compressed into a smaller number of uncorrelated dimensions.
- **Factor Analysis is a Theoretical Explainer.** It seeks to identify latent causal drivers. It explicitly ignores Unique Variance (noise) and focuses only on **Common Variance** (the shared signal between variables).
- In this notebook, we will implement both on a synthetic dataset to see exactly how their philosophies lead to different outputs.

In [ ]:
# 2. Setup and Required Imports
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.decomposition import PCA, FactorAnalysis
from sklearn.preprocessing import StandardScaler

# Set display options and plot style
pd.set_option('display.max_rows', 100)
pd.set_option('display.max_columns', None)
plt.style.use('seaborn-v0_8-whitegrid')

# Set random seed for perfect reproducibility
np.random.seed(42)
print("Libraries successfully imported and environment ready!")

## 3. Data Creation: The "Messy" Survey Dataset
To highlight the difference between PCA (which includes noise) and FA (which excludes noise), we need a dataset with clear latent factors AND significant, explicit noise on a specific variable.

We will simulate a Customer Satisfaction Survey with 6 questions (Q1-Q6) based on 2 Latent Factors: 
1. Product Quality
2. Customer Service

**The Trick:** We will make Q3 extremely noisy. PCA will treat this noise as important variance. FA will recognize it as "Unique Variance" and ignore it.

In [ ]:
n_customers = 500

# True Latent Factors
quality_factor = np.random.normal(0, 1, n_customers)
service_factor = np.random.normal(0, 1, n_customers)

# Q1-Q3 driven by Quality
q1 = 0.8 * quality_factor + np.random.normal(0, 0.2, n_customers)
q2 = 0.85 * quality_factor + np.random.normal(0, 0.2, n_customers)
# Q3 has MASSIVE UNIQUE NOISE
q3 = 0.7 * quality_factor + np.random.normal(0, 2.5, n_customers) 

# Q4-Q6 driven by Service
q4 = 0.9 * service_factor + np.random.normal(0, 0.2, n_customers)
q5 = 0.8 * service_factor + np.random.normal(0, 0.3, n_customers)
q6 = 0.85 * service_factor + np.random.normal(0, 0.2, n_customers)

# Combine into DataFrame
df_survey = pd.DataFrame({
    'Q1_Reliability': q1,
    'Q2_Durability': q2,
    'Q3_Packaging_Design': q3, # The Noisy Variable
    'Q4_Wait_Time': q4,
    'Q5_Staff_Friendly': q5,
    'Q6_Problem_Resolution': q6
})

print("Synthetic Survey Dataset Created.")
print(df_survey.head().round(2))

## 4. Preprocessing: Standardization
- Both PCA and Factor Analysis are highly sensitive to scale. 
- Because variance is the core metric, a variable with a range of 1-1000 will mathematically dominate a variable with a range of 1-5 simply due to numerical scale.
- We must always standardize our variables to have a mean of 0 and a standard deviation of 1 before running these models.

In [ ]:
scaler = StandardScaler()
df_scaled = pd.DataFrame(scaler.fit_transform(df_survey), columns=df_survey.columns)

print("--- Data Standardized ---")
print(f"Mean of Q1: {df_scaled['Q1_Reliability'].mean():.4f} (~0)")
print(f"Std Dev of Q1: {df_scaled['Q1_Reliability'].std():.4f} (~1.0)")

# Let's check the correlation matrix to see the "pockets" of shared variance
plt.figure(figsize=(8, 6))
sns.heatmap(df_scaled.corr(), annot=True, cmap='coolwarm', fmt=".2f", vmin=-1, vmax=1)
plt.title('Correlation Matrix of Survey Variables', fontsize=14)
plt.show()

print("Notice: Q3 has very weak correlations (<0.3) with the rest of the Quality group because its unique noise drowns out the shared signal.")

## 5. Running Principal Component Analysis (PCA)
- Goal: Compress the 6 variables into a smaller set of orthogonal (uncorrelated) components while retaining maximum Total Variance.
- Let's extract 2 Principal Components and look at the "Loadings" (the weights that connect the original variables to the new components).

In [ ]:
# Fit PCA
pca = PCA(n_components=2, random_state=42)
pca.fit(df_scaled)

# Extract Loadings
pca_loadings = pd.DataFrame(
    pca.components_.T,
    columns=['PC1', 'PC2'],
    index=df_scaled.columns
)

print("--- PCA Loadings (Weights) ---")
print(pca_loadings.round(3))

print("\n--- Variance Explained ---")
print(f"PC1 Explains {pca.explained_variance_ratio_[0]*100:.1f}% of Total Variance.")
print(f"PC2 Explains {pca.explained_variance_ratio_[1]*100:.1f}% of Total Variance.")
print(f"Total Information Retained: {np.sum(pca.explained_variance_ratio_)*100:.1f}%")

## 6. Running Factor Analysis (FA)
- Goal: Identify the underlying theoretical drivers by focusing ONLY on Common Variance (shared signal).
- We will extract 2 Factors using 'varimax' rotation (which we learned makes factors more interpretable).

In [ ]:
# Fit Factor Analysis with Varimax Rotation
fa = FactorAnalysis(n_components=2, rotation='varimax', random_state=42)
fa.fit(df_scaled)

# Extract Loadings
fa_loadings = pd.DataFrame(
    fa.components_.T,
    columns=['Factor_1', 'Factor_2'],
    index=df_scaled.columns
)

print("--- Factor Analysis Loadings (Varimax Rotated) ---")
print(fa_loadings.round(3))

## 7. The Philosophical Split: Visualizing the Difference
Let's compare how PCA and FA handled the noisy variable, Q3.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 6))

# Plot PCA Loadings
sns.heatmap(pca_loadings, annot=True, cmap='mako', vmin=-1, vmax=1, ax=axes[0])
axes[0].set_title('PCA: Total Variance Perspective', fontsize=14)

# Plot FA Loadings
sns.heatmap(fa_loadings, annot=True, cmap='mako', vmin=-1, vmax=1, ax=axes[1])
axes[1].set_title('Factor Analysis: Common Variance Perspective', fontsize=14)

plt.tight_layout()
plt.show()

print("CRITICAL OBSERVATION ON Q3 (The Noisy Variable):")
print("- In PCA: Q3 has a moderate loading (~0.4) on PC2. PCA sees its massive unique variance as 'information' and tries to include it in the component.")
print("- In FA: Q3 is almost completely ignored (loadings ~0.2). FA recognizes that Q3's variance is not shared with Q1 or Q2, classifies it as 'Unique Error', and excludes it from the latent construct.")

## 8. The Geometry of Dimensionality Reduction (PCA)
PCA works by rotating the coordinate system to align with the directions of maximum variance. Let's visualize this using two highly correlated variables: Q4 and Q5.

In [ ]:
plt.figure(figsize=(8, 8))
plt.scatter(df_scaled['Q4_Wait_Time'], df_scaled['Q5_Staff_Friendly'], alpha=0.5, color='blue')

# Fit a mini PCA just for these two variables to get the vectors
mini_pca = PCA(n_components=2)
mini_pca.fit(df_scaled[['Q4_Wait_Time', 'Q5_Staff_Friendly']])

# Plot the Principal Components as vectors
for i, (length, vector) in enumerate(zip(mini_pca.explained_variance_, mini_pca.components_)):
    v = vector * 3 * np.sqrt(length)
    plt.plot([0, v[0]], [0, v[1]], '-k', linewidth=3, label=f'PC{i+1}' if i==0 else "")
    if i == 0:
        plt.annotate('PC1 (Max Variance)', (v[0]*1.1, v[1]*1.1), color='black', weight='bold', fontsize=12)
    else:
        plt.annotate('PC2 (Orthogonal)', (v[0]*1.1, v[1]*1.1), color='black', weight='bold', fontsize=12)

plt.title('The Geometry of PCA: Rotating Axes to Capture Variance', fontsize=14)
plt.xlabel('Q4: Wait Time (Standardized)')
plt.ylabel('Q5: Staff Friendly (Standardized)')
plt.axis('equal')
plt.grid(alpha=0.3)
plt.show()

print("Notice how PC1 is drawn directly through the main cloud of data. It captures the vast majority of the information.")
print("PC2 is forced to be exactly 90 degrees (orthogonal) to PC1, capturing the remaining residual spread.")

## 9. Orthogonality in PCA
The requirement that components be orthogonal (90-degree angles) is not just a mathematical convenience. It guarantees that the resulting Principal Components are perfectly UNCORRELATED with each other (r = 0.0).
This makes them ideal for subsequent Machine Learning models to prevent Multicollinearity.

In [ ]:
# Transform the original data into the new PCA coordinate space
pca_scores = pd.DataFrame(pca.transform(df_scaled), columns=['PC1', 'PC2'])

# Check the correlation between PC1 and PC2
corr_pca = pca_scores.corr()

plt.figure(figsize=(6, 5))
sns.heatmap(corr_pca, annot=True, cmap='coolwarm', fmt=".4f", vmin=-1, vmax=1)
plt.title('Correlation Matrix of Principal Components', fontsize=14)
plt.show()

print("Proof of Orthogonality: The correlation between PC1 and PC2 is exactly zero.")
print("They provide perfectly independent, non-overlapping information.")

## 10. The Scree Plot: Determining "How Many" Components to Keep
PCA generates as many components as there are variables (6 in our case).
We use the **Cumulative Explained Variance** and the **Scree Plot** to decide when to stop retaining components.

In [ ]:
# Fit PCA to ALL 6 components
pca_full = PCA(n_components=6)
pca_full.fit(df_scaled)

explained_variance_ratio = pca_full.explained_variance_ratio_
cumulative_variance = np.cumsum(explained_variance_ratio)

plt.figure(figsize=(10, 6))
# Plot individual variance
plt.bar(range(1, 7), explained_variance_ratio * 100, alpha=0.6, color='blue', label='Individual Variance %')
# Plot cumulative variance
plt.plot(range(1, 7), cumulative_variance * 100, marker='o', color='red', linewidth=2, label='Cumulative Variance %')

plt.axhline(y=85, color='green', linestyle='--', label='85% Threshold')
plt.title('Scree Plot and Cumulative Variance', fontsize=15)
plt.xlabel('Principal Component')
plt.ylabel('Percentage of Variance Explained')
plt.legend()
plt.grid(alpha=0.3)
plt.tight_layout()
plt.show()

print("Decision Rule: We typically look for the 'elbow' in the bar chart, or retain enough components to cross an arbitrary threshold (e.g., 85%).")
print("Here, 3 components capture over 85% of the information. We can safely discard PC4, PC5, and PC6, reducing our dataset by 50%.")

## 11. Practice Exercises: Regional Sales KPIs
**Scenario:** You are a data scientist for a retail chain. You have 5 Regional KPIs: 
`Revenue`, `Profit`, `Customer_Footfall`, `Online_Visits`, and `Random_Errors`.

You need to compress this data to feed into a predictive model without multicollinearity.

In [ ]:
# Generate practice data
n_regions = 200
latent_perf = np.random.normal(0, 1, n_regions)

df_retail = pd.DataFrame({
    'Revenue': 0.9 * latent_perf + np.random.normal(0, 0.2, n_regions),
    'Profit': 0.85 * latent_perf + np.random.normal(0, 0.2, n_regions),
    'Footfall': 0.8 * latent_perf + np.random.normal(0, 0.3, n_regions),
    'Online_Visits': 0.7 * latent_perf + np.random.normal(0, 0.4, n_regions),
    'Random_Errors': np.random.normal(0, 2.0, n_regions) # The noise variable
})
print("Practice Retail Dataset Created.")

### Exercise 1: Standardize and run PCA
1. Standardize the `df_retail` data.
2. Run PCA extracting ALL 5 components.
3. Print the percentage of variance explained by PC1.

In [ ]:
# --- EXERCISE 1 SOLUTION ---
# 1. Standardize
scaler_ex = StandardScaler()
retail_scaled = pd.DataFrame(scaler_ex.fit_transform(df_retail), columns=df_retail.columns)

# 2. Run PCA
pca_ex = PCA(n_components=5)
pca_ex.fit(retail_scaled)

# 3. Variance Explained
pc1_var = pca_ex.explained_variance_ratio_[0] * 100
print(f"Exercise 1 Results:")
print(f"PC1 captures {pc1_var:.2f}% of the total variance in the retail data.")

### Exercise 2: The Noise Trap
Look at the PCA loadings for PC2. Does it heavily load onto the `Random_Errors` column? Print the loading value to check.

In [ ]:
# --- EXERCISE 2 SOLUTION ---
pca_loadings_ex = pd.DataFrame(
    pca_ex.components_.T,
    columns=[f'PC{i+1}' for i in range(5)],
    index=retail_scaled.columns
)

pc2_error_loading = pca_loadings_ex.loc['Random_Errors', 'PC2']

print("--- Exercise 2 Results ---")
print(f"The loading of Random_Errors on PC2 is: {pc2_error_loading:.4f}")
print("\nBecause Random_Errors has massive variance, PCA treats it as highly important information and builds an entire component (PC2) mostly around it.")
print("This is the core danger of PCA vs FA. PCA compresses noise just as happily as it compresses signal.")

## 12. Summary and Key Takeaways

- **The Goal:** PCA maximizes Total Variance for data compression. FA models Common Variance for causal explanation.
- **The Output:** PCA creates independent, orthogonal components (Math-based). FA creates interpretable latent factors (Theory-based).
- **Noise Handling:** PCA treats unique noise as information to be captured. FA explicitly identifies and excludes unique noise.
- **Orthogonality:** PCA components are guaranteed to be perfectly uncorrelated (r = 0), making them the perfect input features for predictive Machine Learning models to prevent Multicollinearity.
- **Model Selection:** Use the Scree Plot and Cumulative Variance Explained to decide how many components to retain. Aim for the 'elbow' or a threshold like 80-90% variance captured.

In [ ]:
print("Module 'PCA vs. Factor Analysis' completed successfully.")
print("You now understand the profound philosophical split between data compression and structural discovery!")